In [50]:
#!/usr/bin/env python3
"""periodic_grf_sdf.py — Step 1: Periodic GRF → SDF Pipeline

Generates random 2D periodic unit-cell geometries using Gaussian random fields
and computes their signed distance function (SDF) representation.

Reference: Liu et al., "Toward Signed Distance Function Based Metamaterial
Design," CMAME (2025).
GitHub:    https://github.com/QibangLiu/SDFGeoDesign
"""

from __future__ import annotations

import numpy as np
from numpy.typing import NDArray
from scipy import ndimage
from scipy.interpolate import RegularGridInterpolator
from skimage.measure import find_contours
import shapely
from shapely.geometry import MultiLineString
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [51]:
# ──────────────────────────────────────────────────────────────────────────────
# Tunable parameters (defaults) — aligned with Liu et al. reference impl.
# ──────────────────────────────────────────────────────────────────────────────
GRF_RESOLUTION: int = 64       # Grid resolution for GRF generation
SDF_RESOLUTION: int = 120      # Grid resolution for SDF evaluation
SDF_DOMAIN_PAD: float = 0.1    # Padding beyond [0,1]^2 for the SDF domain
LEN_SCALE: float = 0.15        # Correlation length of Gaussian covariance
VARIANCE: float = 10.0         # Variance (sill) of the GRF
GRF_MODE_NO: int = 32          # Number of Fourier modes per dim (gstools)
THRESHOLD_SCALE: float = 0.4   # Fraction of field range for threshold window
VFRAC_MIN: float = 0.15        # Minimum allowed solid volume fraction
VFRAC_MAX: float = 0.85        # Maximum allowed solid volume fraction
PRINT_TOL: float = 0.08        # Manufacturing tolerance (fraction of domain)
MIN_HOLE_RADIUS: float = 0.04  # Min equivalent radius for interior holes
NARROW_THRESHOLD: float = 0.024  # Min dist between non-adjacent contour pts
MAX_RETRIES: int = 50          # Max consecutive rejections before error
N_SAMPLES: int = 10             # Number of valid unit-cell samples to generate


# ──────────────────────────────────────────────────────────────────────────────
# 1. GRF Generation
# ──────────────────────────────────────────────────────────────────────────────
def generate_periodic_grf(
    resolution: int = GRF_RESOLUTION,
    len_scale: float = LEN_SCALE,
    variance: float = VARIANCE,
    mode_no: int = GRF_MODE_NO,
    rng: np.random.Generator | None = None,
) -> NDArray[np.float64]:
    """Generate a 2D periodic Gaussian random field on [0, 1)^2.

    Uses gstools with the explicit Fourier generator (``generator="Fourier"``,
    ``mode_no=32``) for periodic fields, matching the reference implementation.
    Falls back to a manual spectral method if gstools is unavailable.

    Parameters
    ----------
    resolution : int
        Number of grid points along each axis.
    len_scale : float
        Correlation length of the Gaussian covariance model.
    variance : float
        Variance (sill) of the covariance model.
    mode_no : int
        Number of Fourier modes per dimension (gstools Fourier generator).
    rng : numpy.random.Generator or None
        Random number generator for reproducibility.

    Returns
    -------
    field : ndarray of shape (resolution, resolution)
        Periodic GRF values on a uniform grid.
    """
    if rng is None:
        rng = np.random.default_rng()

    x = np.linspace(0, 1, resolution, endpoint=False)

    try:
        import gstools as gs

        model = gs.Gaussian(dim=2, var=variance, len_scale=len_scale)
        srf = gs.SRF(
            model,
            generator="Fourier",
            period=[1.0, 1.0],
            mode_no=mode_no,
            seed=int(rng.integers(0, 2**31)),
        )
        field = srf.structured([x, x])
        return np.asarray(field, dtype=np.float64)
    except Exception:
        return _generate_periodic_grf_manual(resolution, len_scale, variance, rng)


def _generate_periodic_grf_manual(
    resolution: int,
    len_scale: float,
    variance: float,
    rng: np.random.Generator,
) -> NDArray[np.float64]:
    """Fallback: periodic GRF via the Fourier spectral method.

    Uses the representation:
        U(x) = sum_{k != 0} sqrt(S(k) * dk^2) * (Z1 cos(k.x) + Z2 sin(k.x))
    where k = 2*pi*(n1, n2) for integer n1, n2, dk = 2*pi, and S(k) is the
    power spectrum of the Gaussian covariance C(r) = var * exp(-(r/l)^2):
        S(k) = var * pi * l^2 * exp(-|k|^2 * l^2 / 4)
    """
    x = np.linspace(0, 1, resolution, endpoint=False)
    xx, yy = np.meshgrid(x, x, indexing="ij")

    # Mode cutoff: include modes where S(k) > ~1e-8 * S(0)
    k_max = np.sqrt(4.0 * 8.0 * np.log(10.0)) / len_scale
    n_max = int(np.ceil(k_max / (2.0 * np.pi))) + 1
    n_max = max(n_max, 3)

    dk2 = (2.0 * np.pi) ** 2  # area element in k-space

    field = np.zeros((resolution, resolution), dtype=np.float64)

    for n1 in range(-n_max, n_max + 1):
        for n2 in range(-n_max, n_max + 1):
            if n1 == 0 and n2 == 0:
                continue
            kx = 2.0 * np.pi * n1
            ky = 2.0 * np.pi * n2
            k_sq = kx**2 + ky**2

            # Gaussian power spectrum (2D FT of Gaussian covariance)
            S_k = variance * np.pi * len_scale**2 * np.exp(
                -k_sq * len_scale**2 / 4.0
            )
            amplitude = np.sqrt(S_k * dk2)

            z1 = rng.standard_normal()
            z2 = rng.standard_normal()
            phase = kx * xx + ky * yy
            field += amplitude * (z1 * np.cos(phase) + z2 * np.sin(phase))

    return field

In [52]:
# ──────────────────────────────────────────────────────────────────────────────
# 2. Thresholding — adaptive, following reference implementation
# ──────────────────────────────────────────────────────────────────────────────
def threshold_grf(
    grf: NDArray[np.float64],
    threshold: float | None = None,
    rng: np.random.Generator | None = None,
    threshold_scale: float = THRESHOLD_SCALE,
) -> tuple[NDArray[np.bool_], float]:
    """Threshold a GRF to produce a binary solid/void map.

    The threshold is chosen adaptively relative to the field's value range,
    matching the reference implementation::

        ave   = (max + min) / 2
        scale = threshold_scale * (max - min)
        v     ~ Uniform(ave, ave + scale)

    This ensures the threshold always falls within the field's actual values
    and biases toward removing the lower-value region.

    Parameters
    ----------
    grf : ndarray
        The Gaussian random field.
    threshold : float or None
        If None, drawn adaptively from the field range.
    rng : Generator or None
        Random number generator.
    threshold_scale : float
        Fraction of (max - min) defining the threshold window width.

    Returns
    -------
    binary : ndarray of bool
        True where solid (GRF >= threshold), False where void.
    threshold_used : float
        The threshold value that was applied.
    """
    if rng is None:
        rng = np.random.default_rng()
    if threshold is None:
        field_min = float(grf.min())
        field_max = float(grf.max())
        ave = (field_max + field_min) / 2.0
        scale = threshold_scale * (field_max - field_min)
        threshold = float(rng.uniform(ave, ave + scale))
    binary: NDArray[np.bool_] = grf >= threshold
    return binary, threshold

In [53]:
# ──────────────────────────────────────────────────────────────────────────────
# 3. Contour Extraction
# ──────────────────────────────────────────────────────────────────────────────
def extract_contours(
    grf: NDArray[np.float64],
    threshold: float,
    resolution: int = GRF_RESOLUTION,
) -> list[NDArray[np.float64]]:
    """Extract boundary contours from the GRF at the given threshold level.

    Uses ``skimage.measure.find_contours`` with ``positive_orientation='high'``
    (matching the reference — contours are oriented so that the high side of
    the field is to the right) and converts pixel coordinates to physical
    coordinates in [0, 1)^2.

    Parameters
    ----------
    grf : ndarray of shape (resolution, resolution)
        The raw GRF field.
    threshold : float
        Level at which to extract contours.
    resolution : int
        Grid resolution (used for coordinate scaling).

    Returns
    -------
    contours : list of ndarray
        Each array has shape (M, 2) with (x, y) in physical coordinates.
    """
    raw_contours = find_contours(grf, level=threshold, positive_orientation="high")
    # Pixel index i corresponds to physical coordinate i / resolution
    scale = 1.0 / resolution
    contours = [c * scale for c in raw_contours]
    return contours

In [54]:
# ──────────────────────────────────────────────────────────────────────────────
# 4. Signed Distance Function — Shapely exact distances + GRF interpolation
# ──────────────────────────────────────────────────────────────────────────────
def compute_sdf(
    contours: list[NDArray[np.float64]],
    grf: NDArray[np.float64],
    threshold: float,
    sdf_resolution: int = SDF_RESOLUTION,
    domain_pad: float = SDF_DOMAIN_PAD,
    grf_resolution: int = GRF_RESOLUTION,
) -> NDArray[np.float64]:
    """Compute the signed distance function on an extended grid.

    * **Distance** is computed via Shapely ``MultiLineString.distance``,
      giving exact shortest distance to boundary *line segments* (not just
      sampled contour points), matching the reference implementation.
    * **Sign** is determined by bilinear interpolation of the GRF to the SDF
      grid and comparing against the threshold (solid where GRF >= threshold).
      This is more accurate near the boundary than a nearest-neighbour lookup
      on the 64x64 binary map.

    Parameters
    ----------
    contours : list of ndarray
        Boundary contours in physical coordinates.
    grf : ndarray of shape (grf_resolution, grf_resolution)
        The raw GRF field (used for sign via interpolation).
    threshold : float
        The GRF threshold (solid where GRF >= threshold).
    sdf_resolution : int
        Number of grid points for the SDF grid.
    domain_pad : float
        Padding beyond [0,1]^2 on each side.
    grf_resolution : int
        Resolution of the GRF grid.

    Returns
    -------
    sdf : ndarray of shape (sdf_resolution, sdf_resolution)
        Signed distance values (negative inside solid, positive outside).
    """
    lo = -domain_pad
    hi = 1.0 + domain_pad
    sx = np.linspace(lo, hi, sdf_resolution)
    sy = np.linspace(lo, hi, sdf_resolution)
    sxx, syy = np.meshgrid(sx, sy, indexing="ij")
    grid_points = np.column_stack([sxx.ravel(), syy.ravel()])

    # ── Distance via Shapely ──────────────────────────────────────────────
    # Tile contour line segments periodically (3x3) so distances near the
    # domain boundary are computed correctly.
    offsets = [
        (-1, -1), (-1, 0), (-1, 1),
        (0, -1),  (0, 0),  (0, 1),
        (1, -1),  (1, 0),  (1, 1),
    ]
    all_line_coords: list[list[list[float]]] = []
    for contour in contours:
        if len(contour) < 2:
            continue
        for dx, dy in offsets:
            shifted = contour + np.array([dx, dy], dtype=np.float64)
            all_line_coords.append(shifted.tolist())

    if len(all_line_coords) == 0:
        # No contours — field is entirely solid or void
        is_solid = grf.mean() >= threshold
        return np.full(
            (sdf_resolution, sdf_resolution),
            -1.0 if is_solid else 1.0,
            dtype=np.float64,
        )

    boundary = MultiLineString(all_line_coords)

    # Vectorised distance (shapely >= 2.0)
    try:
        pts_geom = shapely.points(grid_points)
        distances = np.asarray(shapely.distance(pts_geom, boundary))
    except (AttributeError, TypeError):
        # Fallback for older shapely
        from shapely.geometry import Point
        distances = np.array([boundary.distance(Point(p)) for p in grid_points])

    distances = distances.reshape(sdf_resolution, sdf_resolution)

    # ── Sign via bilinear GRF interpolation ───────────────────────────────
    # Pad GRF by one cell (periodic wrap) so interpolation covers [0, 1].
    x_grf = np.linspace(0, 1, grf_resolution, endpoint=False)
    grf_padded = np.pad(grf, ((0, 1), (0, 1)), mode="wrap")
    x_padded = np.append(x_grf, 1.0)

    interp = RegularGridInterpolator(
        (x_padded, x_padded),
        grf_padded,
        method="linear",
        bounds_error=False,
        fill_value=None,
    )
    wrapped = np.column_stack([sxx.ravel() % 1.0, syy.ravel() % 1.0])
    grf_interp = interp(wrapped).reshape(sdf_resolution, sdf_resolution)
    is_solid = grf_interp >= threshold

    # Negative inside solid, positive outside
    sdf = np.where(is_solid, -distances, distances)
    return sdf

In [55]:
# ──────────────────────────────────────────────────────────────────────────────
# 5. Validation Filters — aligned with reference implementation
# ──────────────────────────────────────────────────────────────────────────────
def check_periodic_connectivity(binary: NDArray[np.bool_]) -> bool:
    """Check that all solid pixels form a single connected component under
    periodic (wrapping) boundary conditions.

    Tiles the binary map 3x3, labels connected components, and verifies that
    every solid pixel in the centre tile has the same label.
    """
    tiled = np.tile(binary, (3, 3))
    labeled, num_features = ndimage.label(tiled)
    if num_features == 0:
        return False
    n = binary.shape[0]
    center = labeled[n : 2 * n, n : 2 * n]
    solid_labels = center[binary]
    if len(solid_labels) == 0:
        return False
    return len(np.unique(solid_labels)) == 1


def _contour_arc_length(contour: NDArray[np.float64]) -> float:
    """Compute the total arc length of a contour."""
    diffs = np.diff(contour, axis=0)
    return float(np.sqrt((diffs**2).sum(axis=1)).sum())


def _contour_is_closed(contour: NDArray[np.float64], tol: float = 1e-3) -> bool:
    """Check whether first and last points of a contour coincide."""
    return bool(np.linalg.norm(contour[0] - contour[-1]) < tol)


def _contour_enclosed_area(contour: NDArray[np.float64]) -> float:
    """Enclosed area via the shoelace formula (absolute value)."""
    x, y = contour[:, 0], contour[:, 1]
    return float(
        0.5 * np.abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))
    )


def _contour_touches_boundary(
    contour: NDArray[np.float64],
    tol: float | None = None,
    resolution: int = GRF_RESOLUTION,
) -> bool:
    """Check if any point of a contour is within *tol* of the [0, 1) boundary."""
    if tol is None:
        tol = 1.5 / resolution
    return bool(contour.min() < tol or contour.max() > 1.0 - tol)


def _has_narrow_regions(
    contours: list[NDArray[np.float64]],
    threshold: float = NARROW_THRESHOLD,
    min_index_gap: int = 5,
) -> bool:
    """Detect narrow regions: non-adjacent contour points closer than *threshold*.

    For points on the **same** contour they must be separated by at least
    *min_index_gap* indices (accounting for periodic wrapping of the contour)
    to be considered non-adjacent.  For points on **different** contours any
    close pair triggers rejection.  Mirrors the reference ``has_narrow_regions``.
    """
    from scipy.spatial import cKDTree

    for i, c1 in enumerate(contours):
        if len(c1) < 2 * min_index_gap:
            continue
        for j in range(i, len(contours)):
            c2 = contours[j]
            if len(c2) < 2:
                continue
            if i == j:
                # Same contour — check non-adjacent pairs
                tree = cKDTree(c1)
                for k in range(len(c1)):
                    idxs = tree.query_ball_point(c1[k], threshold)
                    for m in idxs:
                        gap = abs(k - m)
                        gap = min(gap, len(c1) - gap)  # periodic gap
                        if gap > min_index_gap:
                            return True
            else:
                # Different contours — any close pair
                tree = cKDTree(c2)
                dists, _ = tree.query(c1)
                if dists.min() < threshold:
                    return True
    return False


def validate_sample(
    binary: NDArray[np.bool_],
    contours: list[NDArray[np.float64]],
    resolution: int = GRF_RESOLUTION,
    vfrac_min: float = VFRAC_MIN,
    vfrac_max: float = VFRAC_MAX,
    print_tol: float = PRINT_TOL,
    min_hole_radius: float = MIN_HOLE_RADIUS,
    narrow_threshold: float = NARROW_THRESHOLD,
) -> bool:
    """Apply validity filters to a generated sample.

    Checks (aligned with the reference where applicable):
    1. Volume fraction within [vfrac_min, vfrac_max].
    2. Single connected solid region (periodic connectivity).
    3. Closed contours too close to domain boundary -> reject.
    4. Interior holes with equivalent radius < min_hole_radius -> reject.
    5. Narrow regions detected between non-adjacent contour points -> reject.

    Parameters
    ----------
    binary : ndarray of bool
        Binary solid/void map.
    contours : list of ndarray
        Extracted boundary contours in physical coordinates.
    resolution : int
        GRF grid resolution.
    vfrac_min, vfrac_max : float
        Allowed volume fraction bounds.
    print_tol : float
        Manufacturing tolerance (domain fraction).
    min_hole_radius : float
        Minimum equivalent radius for interior holes.
    narrow_threshold : float
        Minimum allowed distance between non-adjacent contour points.

    Returns
    -------
    valid : bool
    """
    # 1. Volume fraction
    vfrac = binary.mean()
    if vfrac < vfrac_min or vfrac > vfrac_max:
        return False

    # 2. Periodic connectivity
    if not check_periodic_connectivity(binary):
        return False

    if len(contours) == 0:
        return False

    # 3 & 4. Per-contour checks
    boundary_tol = 0.5 * print_tol  # reference uses fac = 0.5 for closed contours
    for c in contours:
        if _contour_is_closed(c):
            # Reject if a closed contour sits too close to the domain boundary
            if _contour_touches_boundary(c, tol=boundary_tol):
                return False
            # Reject if the enclosed feature is too small
            area = _contour_enclosed_area(c)
            equiv_r = np.sqrt(area / np.pi)
            if equiv_r < min_hole_radius:
                return False

    # 5. Narrow region detection
    if _has_narrow_regions(contours, threshold=narrow_threshold):
        return False

    return True

In [56]:
# ──────────────────────────────────────────────────────────────────────────────
# 6. Single sample + batch generation
# ──────────────────────────────────────────────────────────────────────────────
def generate_single_sample(
    rng: np.random.Generator,
    resolution: int = GRF_RESOLUTION,
    sdf_resolution: int = SDF_RESOLUTION,
    len_scale: float = LEN_SCALE,
    variance: float = VARIANCE,
    mode_no: int = GRF_MODE_NO,
    threshold_scale: float = THRESHOLD_SCALE,
    domain_pad: float = SDF_DOMAIN_PAD,
) -> dict | None:
    """Generate one sample.  Returns None if validation fails.

    Returns
    -------
    sample : dict or None
        Keys: grf, binary, contours, sdf, volume_fraction, threshold.
    """
    grf = generate_periodic_grf(resolution, len_scale, variance, mode_no, rng)
    binary, thresh = threshold_grf(grf, rng=rng, threshold_scale=threshold_scale)
    contours = extract_contours(grf, thresh, resolution)

    if not validate_sample(binary, contours, resolution):
        return None

    sdf = compute_sdf(contours, grf, thresh, sdf_resolution, domain_pad, resolution)

    return {
        "grf": grf,
        "binary": binary,
        "contours": contours,
        "sdf": sdf,
        "volume_fraction": float(binary.mean()),
        "threshold": thresh,
    }


def generate_dataset(
    n_samples: int,
    seed: int = 42,
    resolution: int = GRF_RESOLUTION,
    sdf_resolution: int = SDF_RESOLUTION,
    len_scale: float = LEN_SCALE,
    variance: float = VARIANCE,
    mode_no: int = GRF_MODE_NO,
    threshold_scale: float = THRESHOLD_SCALE,
    domain_pad: float = SDF_DOMAIN_PAD,
    max_retries: int = MAX_RETRIES,
) -> dict:
    """Generate a dataset of valid periodic unit-cell SDF samples.

    Parameters
    ----------
    n_samples : int
        Number of valid samples to produce.
    seed : int
        Random seed for reproducibility.
    resolution : int
        GRF grid resolution.
    sdf_resolution : int
        SDF grid resolution.
    len_scale : float
        GRF correlation length.
    variance : float
        GRF variance.
    mode_no : int
        Number of Fourier modes for gstools.
    threshold_scale : float
        Fraction of field range for adaptive threshold window.
    domain_pad : float
        SDF domain padding beyond [0,1]^2.
    max_retries : int
        Maximum consecutive rejections before raising RuntimeError.

    Returns
    -------
    dataset : dict
        "sdf"             : ndarray (n_samples, sdf_resolution, sdf_resolution)
        "binary"          : ndarray (n_samples, resolution, resolution)
        "contours"        : list[list[ndarray]]
        "volume_fraction" : ndarray (n_samples,)
        "threshold"       : ndarray (n_samples,)
        "grf"             : ndarray (n_samples, resolution, resolution)
        "n_rejected"      : int — total number of rejected samples
    """
    rng = np.random.default_rng(seed)

    sdfs: list[NDArray] = []
    binaries: list[NDArray] = []
    all_contours: list[list[NDArray]] = []
    vfracs: list[float] = []
    thresholds: list[float] = []
    grfs: list[NDArray] = []
    n_rejected = 0
    consecutive_rejects = 0

    while len(sdfs) < n_samples:
        sample = generate_single_sample(
            rng, resolution, sdf_resolution, len_scale, variance,
            mode_no, threshold_scale, domain_pad,
        )
        if sample is None:
            n_rejected += 1
            consecutive_rejects += 1
            if consecutive_rejects >= max_retries:
                raise RuntimeError(
                    f"Failed after {max_retries} consecutive rejections. "
                    f"Generated {len(sdfs)}/{n_samples} samples so far. "
                    f"Consider adjusting len_scale, threshold_scale, or filters."
                )
            continue

        consecutive_rejects = 0
        sdfs.append(sample["sdf"])
        binaries.append(sample["binary"])
        all_contours.append(sample["contours"])
        vfracs.append(sample["volume_fraction"])
        thresholds.append(sample["threshold"])
        grfs.append(sample["grf"])

        print(
            f"  [{len(sdfs):3d}/{n_samples}] "
            f"VF={sample['volume_fraction']:.3f}  "
            f"thr={sample['threshold']:+.3f}  "
            f"(rejected so far: {n_rejected})"
        )

    return {
        "sdf": np.stack(sdfs),
        "binary": np.stack(binaries),
        "contours": all_contours,
        "volume_fraction": np.array(vfracs),
        "threshold": np.array(thresholds),
        "grf": np.stack(grfs),
        "n_rejected": n_rejected,
    }

In [57]:
# ──────────────────────────────────────────────────────────────────────────────
# 7. Visualization
# ──────────────────────────────────────────────────────────────────────────────
from mpl_toolkits.axes_grid1 import make_axes_locatable


def _add_colorbar(ax, im, label: str = "", size: str = "4%", pad: float = 0.04):
    """Append a thin colorbar to *ax* without resizing the image."""
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size=size, pad=pad)
    cb = plt.colorbar(im, cax=cax)
    if label:
        cb.set_label(label, fontsize=8)
    cb.ax.tick_params(labelsize=7)
    return cb


def _plot_polyline_contours(
    ax,
    contours: list[NDArray[np.float64]],
    xlim: tuple[float, float],
    ylim: tuple[float, float] | None = None,
    periodic: bool = False,
    color: str = "red",
    linewidth: float = 1.2,
) -> None:
    """Draw contour polylines directly in physical coordinates."""
    if ylim is None:
        ylim = xlim
    offsets = [
        (-1.0, -1.0), (-1.0, 0.0), (-1.0, 1.0),
        (0.0, -1.0),  (0.0, 0.0),  (0.0, 1.0),
        (1.0, -1.0),  (1.0, 0.0),  (1.0, 1.0),
    ] if periodic else [(0.0, 0.0)]

    for contour in contours:
        contour = np.asarray(contour, dtype=np.float64)
        if len(contour) < 2:
            continue
        for dx, dy in offsets:
            shifted = contour + np.array([dx, dy], dtype=np.float64)
            if shifted[:, 0].max() < xlim[0] or shifted[:, 0].min() > xlim[1]:
                continue
            if shifted[:, 1].max() < ylim[0] or shifted[:, 1].min() > ylim[1]:
                continue
            ax.plot(shifted[:, 0], shifted[:, 1], color=color, linewidth=linewidth)


def plot_sdf_gallery(
    dataset: dict,
    sdf_resolution: int = SDF_RESOLUTION,
    domain_pad: float = SDF_DOMAIN_PAD,
    filename: str = "sdf_gallery.png",
) -> None:
    """4x4 gallery of SDF fields with zero contours overlaid in red."""
    n = min(16, dataset["sdf"].shape[0])
    fig, axes = plt.subplots(4, 4, figsize=(16, 14))
    lo, hi = -domain_pad, 1.0 + domain_pad
    has_contours = "contours" in dataset

    for idx in range(16):
        ax = axes[idx // 4, idx % 4]
        if idx < n:
            sdf = dataset["sdf"][idx]
            vf = dataset["volume_fraction"][idx]
            vmax = np.abs(sdf).max()
            im = ax.imshow(
                sdf.T, origin="lower", extent=[lo, hi, lo, hi],
                cmap="RdBu", vmin=-vmax, vmax=vmax,
            )
            if has_contours:
                _plot_polyline_contours(
                    ax,
                    dataset["contours"][idx],
                    xlim=(lo, hi),
                    periodic=True,
                    linewidth=1.2,
                )
            else:
                ax.contour(
                    np.linspace(lo, hi, sdf.shape[0]),
                    np.linspace(lo, hi, sdf.shape[1]),
                    sdf.T, levels=[0.0], colors="red", linewidths=1.2,
                )
            ax.set_title(f"VF = {vf:.3f}", fontsize=9)
            ax.set_xlim(lo, hi)
            ax.set_ylim(lo, hi)
            ax.set_aspect("equal")
            _add_colorbar(ax, im, label="SDF")
        else:
            ax.axis("off")
        ax.set_xticks([])
        ax.set_yticks([])

    fig.suptitle("SDF Gallery (zero contour in red)", fontsize=14)
    fig.tight_layout()
    fig.savefig(filename, dpi=150)
    plt.close(fig)
    print(f"Saved {filename}")


def plot_grf_gallery(dataset: dict, filename: str = "grf_gallery.png") -> None:
    """4x4 gallery of raw GRF fields with threshold contours overlaid."""
    n = min(16, dataset["grf"].shape[0])
    fig, axes = plt.subplots(4, 4, figsize=(16, 14))
    has_contours = "contours" in dataset

    for idx in range(16):
        ax = axes[idx // 4, idx % 4]
        if idx < n:
            grf = dataset["grf"][idx]
            thresh = dataset["threshold"][idx]
            res = grf.shape[0]
            im = ax.imshow(grf.T, origin="lower", extent=[0, 1, 0, 1],
                           cmap="viridis")
            if has_contours:
                _plot_polyline_contours(
                    ax,
                    dataset["contours"][idx],
                    xlim=(0.0, 1.0),
                    linewidth=1.2,
                )
            else:
                x = np.linspace(0.0, 1.0, res, endpoint=False)
                ax.contour(
                    x,
                    x,
                    grf.T, levels=[thresh], colors="red", linewidths=1.2,
                )
            ax.set_title(f"thr = {thresh:+.3f}", fontsize=9)
            ax.set_aspect("equal")
            _add_colorbar(ax, im, label="GRF")
        else:
            ax.axis("off")
        ax.set_xticks([])
        ax.set_yticks([])

    fig.suptitle("GRF Gallery (threshold contour in red)", fontsize=14)
    fig.tight_layout()
    fig.savefig(filename, dpi=150)
    plt.close(fig)
    print(f"Saved {filename}")


def plot_sample_detail(
    dataset: dict,
    sample_idx: int = 0,
    sdf_resolution: int = SDF_RESOLUTION,
    domain_pad: float = SDF_DOMAIN_PAD,
    filename: str = "sample_detail.png",
) -> None:
    """Detailed 4-panel figure for a single sample."""
    grf = dataset["grf"][sample_idx]
    binary = dataset["binary"][sample_idx]
    sdf = dataset["sdf"][sample_idx]
    thresh = dataset["threshold"][sample_idx]
    vf = dataset["volume_fraction"][sample_idx]
    res = grf.shape[0]
    has_contours = "contours" in dataset

    lo, hi = -domain_pad, 1.0 + domain_pad
    vmax = np.abs(sdf).max()

    fig, axes = plt.subplots(1, 4, figsize=(24, 5))

    # (a) Raw GRF with threshold contour
    ax = axes[0]
    im = ax.imshow(grf.T, origin="lower", extent=[0, 1, 0, 1], cmap="viridis")
    if has_contours:
        _plot_polyline_contours(
            ax,
            dataset["contours"][sample_idx],
            xlim=(0.0, 1.0),
            linewidth=1.5,
        )
    else:
        x = np.linspace(0.0, 1.0, res, endpoint=False)
        ax.contour(
            x, x,
            grf.T, levels=[thresh], colors="red", linewidths=1.5,
        )
    ax.set_title(f"(a) GRF  (thr = {thresh:+.3f})", fontsize=10)
    ax.set_aspect("equal")
    _add_colorbar(ax, im, label="GRF value")

    # (b) Binary solid/void
    ax = axes[1]
    im = ax.imshow(binary.T.astype(float), origin="lower", extent=[0, 1, 0, 1],
                   cmap="gray_r", vmin=0, vmax=1)
    ax.set_title(f"(b) Binary  (VF = {vf:.3f})", fontsize=10)
    ax.set_aspect("equal")
    cb = _add_colorbar(ax, im)
    cb.set_ticks([0, 1])
    cb.set_ticklabels(["Void", "Solid"])

    # (c) SDF with zero contour
    ax = axes[2]
    im = ax.imshow(
        sdf.T, origin="lower", extent=[lo, hi, lo, hi],
        cmap="RdBu", vmin=-vmax, vmax=vmax,
    )
    if has_contours:
        _plot_polyline_contours(
            ax,
            dataset["contours"][sample_idx],
            xlim=(lo, hi),
            periodic=True,
            linewidth=1.5,
        )
    else:
        ax.contour(
            np.linspace(lo, hi, sdf.shape[0]),
            np.linspace(lo, hi, sdf.shape[1]),
            sdf.T, levels=[0.0], colors="red", linewidths=1.5,
        )
    ax.set_title("(c) SDF  (zero contour)", fontsize=10)
    ax.set_aspect("equal")
    _add_colorbar(ax, im, label="Signed distance")

    # (d) Tiled 3x3 SDF for periodicity verification
    ax = axes[3]
    sx = np.linspace(lo, hi, sdf.shape[0])
    sy = np.linspace(lo, hi, sdf.shape[1])
    mask_x = (sx >= 0.0) & (sx <= 1.0)
    mask_y = (sy >= 0.0) & (sy <= 1.0)
    sdf_unit = sdf[np.ix_(mask_x, mask_y)]
    sdf_tiled = np.tile(sdf_unit, (3, 3))
    im = ax.imshow(
        sdf_tiled.T, origin="lower", extent=[0, 3, 0, 3],
        cmap="RdBu", vmin=-vmax, vmax=vmax,
    )
    for i in range(1, 3):
        ax.axhline(i, color="k", linewidth=0.5, linestyle="--")
        ax.axvline(i, color="k", linewidth=0.5, linestyle="--")
    ax.set_title("(d) SDF tiled 3x3  (periodicity)", fontsize=10)
    ax.set_aspect("equal")
    _add_colorbar(ax, im, label="Signed distance")

    fig.suptitle(f"Sample {sample_idx} -- Detailed View", fontsize=13, y=1.02)
    fig.tight_layout()
    fig.savefig(filename, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {filename}")

In [58]:
print("="*60)
print("Generating periodic GRF → SDF dataset")
print(f"  len_scale={LEN_SCALE}, variance={VARIANCE}, threshold_scale={THRESHOLD_SCALE}")
print(f"  GRF resolution={GRF_RESOLUTION}, SDF resolution={SDF_RESOLUTION}")
print(f"  Volume fraction range: [{VFRAC_MIN}, {VFRAC_MAX}]")
print()

dataset = generate_dataset(n_samples=N_SAMPLES, seed=42)

n_valid = dataset["sdf"].shape[0]
n_rejected = dataset["n_rejected"]
vf = dataset["volume_fraction"]
print(f"\n{'='*60}")
print(f"Dataset complete: {n_valid} valid samples ({n_rejected} rejected)")
print(f"  Volume fractions — mean: {vf.mean():.3f}")
print(f"                    std:  {vf.std():.3f}")
print(f"                    min:  {vf.min():.3f}, max: {vf.max():.3f}")
print()

plot_sdf_gallery(dataset, filename="sdf_gallery.png")
plot_grf_gallery(dataset, filename="grf_gallery.png")
plot_sample_detail(dataset, sample_idx=0, filename="sample_detail.png")

print("\nDone.")

Generating periodic GRF → SDF dataset
  len_scale=0.15, variance=10.0, threshold_scale=0.4
  GRF resolution=64, SDF resolution=120
  Volume fraction range: [0.15, 0.85]

  [  1/10] VF=0.629  thr=-0.358  (rejected so far: 39)
  [  2/10] VF=0.766  thr=-1.338  (rejected so far: 80)
  [  3/10] VF=0.800  thr=-0.929  (rejected so far: 97)
  [  4/10] VF=0.612  thr=-2.185  (rejected so far: 113)
  [  5/10] VF=0.453  thr=+1.656  (rejected so far: 134)
  [  6/10] VF=0.648  thr=-0.705  (rejected so far: 168)
  [  7/10] VF=0.752  thr=-2.055  (rejected so far: 173)
  [  8/10] VF=0.566  thr=+0.876  (rejected so far: 183)
  [  9/10] VF=0.582  thr=-1.515  (rejected so far: 214)
  [ 10/10] VF=0.479  thr=+1.668  (rejected so far: 216)

Dataset complete: 10 valid samples (216 rejected)
  Volume fractions — mean: 0.629
                    std:  0.111
                    min:  0.453, max: 0.800

Saved sdf_gallery.png
Saved grf_gallery.png
Saved sample_detail.png

Done.


In [59]:
# Save dataset to NPZ for downstream pipeline
import os

DATA_DIR = "data"
DATASET_FILENAME = "unit_cell_dataset.npz"
DATASET_PATH = os.path.join(DATA_DIR, DATASET_FILENAME)
os.makedirs(DATA_DIR, exist_ok=True)

np.savez_compressed(
    DATASET_PATH,
    sdf=dataset["sdf"],
    binary=dataset["binary"],
    grf=dataset["grf"],
    volume_fraction=dataset["volume_fraction"],
    threshold=dataset["threshold"],
    contours=np.array(dataset["contours"], dtype=object),
)
print(f"Saved dataset to {DATASET_PATH}")
print(f"  Shape: sdf={dataset['sdf'].shape}, grf={dataset['grf'].shape}")

Saved dataset to data/unit_cell_dataset.npz
  Shape: sdf=(10, 120, 120), grf=(10, 64, 64)


In [60]:
# ──────────────────────────────────────────────────────────────────────────────
# 8. Mesh Pipeline — Imports & Constants
# ──────────────────────────────────────────────────────────────────────────────
import os
import sys
import gmsh
import meshio
from shapely.geometry import LineString, Polygon, Point, MultiPolygon, box
from shapely.ops import unary_union, polygonize

# Output directories
MESH_DIR = "meshes"
DATA_DIR = "data"
DATASET_FILENAME = "unit_cell_dataset.npz"
DATASET_PATH = os.path.join(DATA_DIR, DATASET_FILENAME)
os.makedirs(MESH_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

# Meshing parameters
MESH_SIZE: float = 0.04          # Target element size (matches reference)
CONTOUR_DECIMATE_TOL: float = 0.005  # RDP simplification tolerance

print(f"Gmsh version : {gmsh.GMSH_API_VERSION}")
print(f"Mesh size    : {MESH_SIZE}")
print(f"Output dirs  : {DATA_DIR}/, {MESH_DIR}/")
print(f"Dataset path : {DATASET_PATH}")

Gmsh version : 4.15.1
Mesh size    : 0.04
Output dirs  : data/, meshes/
Dataset path : data/unit_cell_dataset.npz


In [61]:
# ──────────────────────────────────────────────────────────────────────────────
# 9. Periodic Contour Extraction & Solid Polygon Construction
# ──────────────────────────────────────────────────────────────────────────────
# Strategy:
#   1. Pad the GRF by one row/column (wrap mode) so find_contours sees periodicity
#   2. Merge contour fragments that meet at domain corners (on a torus, all 4
#      corners are the same point)
#   3. Use Shapely's `polygonize` to split [0,1]^2 into regions bounded by
#      contour lines, then classify each region as solid or void via the GRF
# ──────────────────────────────────────────────────────────────────────────────

# Domain corners
CORNERS = np.array([[0.0, 0.0], [1.0, 0.0], [1.0, 1.0], [0.0, 1.0]])


def _near_any_corner(pt: NDArray, tol: float = 0.025) -> bool:
    """Check if a point is near any of the 4 domain corners."""
    for c in CORNERS:
        if np.linalg.norm(pt - c) < tol:
            return True
    return False


def extract_periodic_contours(
    grf: NDArray[np.float64],
    threshold: float,
    resolution: int = GRF_RESOLUTION,
) -> list[NDArray[np.float64]]:
    """Extract contours from a wrap-padded GRF for correct periodic boundary handling.

    The GRF is padded by one row/column with wrap mode (65×65 for 64×64 input),
    so ``find_contours`` can trace through the periodic boundary.  After extraction
    we merge contour fragments that meet at equivalent domain corners and filter
    tiny corner artifacts.

    Parameters
    ----------
    grf : ndarray(N, N)
        Raw GRF field (square).
    threshold : float
        Level-set value.
    resolution : int
        Grid resolution (N).

    Returns
    -------
    contours : list of ndarray(M, 2)
        Contours in [0, 1]^2 physical coordinates.
    """
    # Pad to 65×65 so contours cross the periodic seam
    grf_padded = np.pad(grf, ((0, 1), (0, 1)), mode="wrap")
    raw = find_contours(grf_padded, level=threshold, positive_orientation="high")
    contours = [c / resolution for c in raw]  # pixel -> physical [0,1]

    # ── Merge fragments at corners (all 4 corners = same point on torus) ──
    merged = True
    result: list[NDArray | None] = [c.copy() for c in contours]
    while merged:
        merged = False
        for i in range(len(result)):
            if result[i] is None:
                continue
            if not _near_any_corner(result[i][-1]):
                continue
            for j in range(len(result)):
                if j == i or result[j] is None:
                    continue
                if _near_any_corner(result[j][0]):
                    result[i] = np.concatenate([result[i], result[j]], axis=0)
                    result[j] = None
                    merged = True
                    break
            if merged:
                break
    out: list[NDArray] = [c for c in result if c is not None]

    # Filter tiny corner artifacts (both endpoints at corners, <= 4 points)
    filtered = []
    for c in out:
        if len(c) <= 4 and _near_any_corner(c[0]) and _near_any_corner(c[-1]):
            continue  # artifact
        filtered.append(c)
    return filtered


def build_solid_polygons(
    grf: NDArray[np.float64],
    threshold: float,
    resolution: int = GRF_RESOLUTION,
    decimate_tol: float = CONTOUR_DECIMATE_TOL,
) -> list:
    """Build valid solid-region Shapely polygons from periodic contours.

    Uses ``shapely.ops.polygonize`` to split the unit cell into closed regions
    bounded by contour lines and the domain boundary, then classifies each
    region as solid or void using the GRF interpolant.

    Parameters
    ----------
    grf : ndarray(N, N)
        Raw GRF field.
    threshold : float
        Level-set value.

    Returns
    -------
    solid_polys : list of shapely.geometry.Polygon
        Valid polygons covering the solid region of [0,1]^2.
    """
    contours = extract_periodic_contours(grf, threshold, resolution)

    # GRF interpolant for solid/void classification
    x_grf = np.linspace(0, 1, grf.shape[0], endpoint=False)
    grf_pad = np.pad(grf, ((0, 1), (0, 1)), mode="wrap")
    x_pad = np.append(x_grf, 1.0)
    interp = RegularGridInterpolator(
        (x_pad, x_pad), grf_pad, method="linear",
        bounds_error=False, fill_value=None,
    )

    def is_solid(pt):
        return float(interp(np.array(pt).reshape(1, -1))[0]) >= threshold

    # Unit cell boundary
    cell = box(0, 0, 1, 1)

    # Simplify contours with Ramer-Douglas-Peucker
    simplified_lines = []
    for c in contours:
        if len(c) < 3:
            continue
        ls = LineString(c)
        s = ls.simplify(decimate_tol, preserve_topology=True)
        simplified_lines.append(s)

    if not simplified_lines:
        # No contours → entire domain is solid or void
        if is_solid([0.5, 0.5]):
            return [cell]
        return []

    # Combine contour lines with the cell boundary and polygonize
    all_lines = [cell.boundary] + simplified_lines
    merged_lines = unary_union(all_lines)
    polys = list(polygonize(merged_lines))

    if not polys:
        # Fallback: no regions found
        if is_solid([0.5, 0.5]):
            return [cell]
        return []

    # Classify each polygon as solid or void
    solid_polys = []
    for poly in polys:
        if not poly.is_valid:
            poly = poly.buffer(0)
        if poly.is_empty or poly.area < 1e-6:
            continue
        rp = poly.representative_point()
        if is_solid([rp.x, rp.y]):
            solid_polys.append(poly)

    return solid_polys


print("Periodic contour extraction & solid polygon builder defined.")

Periodic contour extraction & solid polygon builder defined.


In [62]:
# ──────────────────────────────────────────────────────────────────────────────
# 10. Gmsh Meshing with Periodic Boundary Constraints
# ──────────────────────────────────────────────────────────────────────────────
# Takes Shapely solid polygons → Gmsh geometry → periodic mesh → .msh file.
# Boundary line segments are tracked for periodic pairing (bottom↔top, left↔right).
# ──────────────────────────────────────────────────────────────────────────────

EDGE_BOTTOM, EDGE_RIGHT, EDGE_TOP, EDGE_LEFT = 0, 1, 2, 3


def _identify_edge(pt: NDArray, tol: float = 0.01) -> tuple[int, float]:
    """Classify a boundary point to an edge and return (edge_id, shared_coord)."""
    x, y = pt[0], pt[1]
    # Corners first
    if abs(x) < tol and abs(y) < tol:
        return EDGE_BOTTOM, 0.0
    if abs(x - 1) < tol and abs(y) < tol:
        return EDGE_RIGHT, 0.0
    if abs(x - 1) < tol and abs(y - 1) < tol:
        return EDGE_TOP, 0.0
    if abs(x) < tol and abs(y - 1) < tol:
        return EDGE_LEFT, 0.0
    # Interior of edges
    if abs(y) < tol:
        return EDGE_BOTTOM, x
    if abs(x - 1) < tol:
        return EDGE_RIGHT, y
    if abs(y - 1) < tol:
        return EDGE_TOP, 1.0 - x
    if abs(x) < tol:
        return EDGE_LEFT, 1.0 - y
    raise ValueError(f"Point {pt} not on any domain boundary (tol={tol})")


def mesh_single_sample(
    grf: NDArray[np.float64],
    threshold: float,
    sample_idx: int,
    mesh_size: float = MESH_SIZE,
    output_dir: str = MESH_DIR,
) -> str | None:
    """Build a periodic Gmsh mesh for one sample.

    Steps:
      1. Build solid polygons via ``build_solid_polygons``
      2. Convert polygon exterior/interior rings to Gmsh line loops
      3. Track boundary-edge segments for periodic constraints
      4. Set periodic pairing (bottom↔top, left↔right)
      5. Generate 2-D triangular mesh and write .msh

    Returns the path to the .msh file, or None on failure.
    """
    solid_polys = build_solid_polygons(grf, threshold)
    if not solid_polys:
        return None

    os.makedirs(output_dir, exist_ok=True)
    msh_path = os.path.join(output_dir, f"sample_{sample_idx:04d}.msh")

    gmsh.initialize()
    gmsh.option.setNumber("General.Verbosity", 0)
    gmsh.model.add("unit_cell")

    try:
        point_cache: dict[tuple[float, float], int] = {}
        # boundary_curves[edge_id] = [(gmsh_line_tag, coord_start, coord_end)]
        boundary_curves: dict[int, list[tuple[int, float, float]]] = {
            e: [] for e in range(4)
        }

        def add_point(x: float, y: float) -> int:
            """Add a point (or return existing) from the cache."""
            key = (round(x, 6), round(y, 6))
            if key not in point_cache:
                point_cache[key] = gmsh.model.geo.addPoint(x, y, 0.0, mesh_size)
            return point_cache[key]

        all_surfaces: list[int] = []

        for poly in solid_polys:
            # Handle MultiPolygon from buffer(0) fixes
            geoms = list(poly.geoms) if poly.geom_type == "MultiPolygon" else [poly]

            for geom in geoms:
                ext_coords = np.array(geom.exterior.coords)
                # Add exterior points (skip closing duplicate)
                ext_tags = [add_point(float(x), float(y)) for x, y in ext_coords[:-1]]
                if len(ext_tags) < 3:
                    continue

                # Create exterior line loop
                ext_curves: list[int] = []
                n = len(ext_tags)
                for k in range(n):
                    p1, p2 = ext_tags[k], ext_tags[(k + 1) % n]
                    if p1 == p2:
                        continue
                    line = gmsh.model.geo.addLine(p1, p2)
                    ext_curves.append(line)

                    # Track boundary segments for periodic constraints
                    c1 = ext_coords[k]
                    c2 = ext_coords[(k + 1) % n]
                    try:
                        e1, _ = _identify_edge(c1)
                        e2, _ = _identify_edge(c2)
                        if e1 == e2:
                            sc = ([c1[0], c2[0]] if e1 in (EDGE_BOTTOM, EDGE_TOP)
                                  else [c1[1], c2[1]])
                            boundary_curves[e1].append(
                                (line, min(sc), max(sc))
                            )
                    except ValueError:
                        pass

                if len(ext_curves) < 3:
                    continue
                try:
                    ext_loop = gmsh.model.geo.addCurveLoop(ext_curves)
                except Exception:
                    continue

                # Interior rings (holes in this polygon)
                hole_loops: list[int] = []
                for interior in geom.interiors:
                    int_coords = np.array(interior.coords)
                    int_tags = [add_point(float(x), float(y))
                                for x, y in int_coords[:-1]]
                    int_curves = []
                    m = len(int_tags)
                    for k in range(m):
                        p1, p2 = int_tags[k], int_tags[(k + 1) % m]
                        if p1 == p2:
                            continue
                        int_curves.append(gmsh.model.geo.addLine(p1, p2))
                    if len(int_curves) >= 3:
                        try:
                            hole_loops.append(
                                gmsh.model.geo.addCurveLoop(int_curves)
                            )
                        except Exception:
                            pass

                try:
                    surf = gmsh.model.geo.addPlaneSurface([ext_loop] + hole_loops)
                    all_surfaces.append(surf)
                except Exception:
                    continue

        if not all_surfaces:
            gmsh.finalize()
            return None

        gmsh.model.geo.synchronize()

        # Physical group for FEniCSx
        all_surfs = [e[1] for e in gmsh.model.getEntities(dim=2)]
        if all_surfs:
            gmsh.model.addPhysicalGroup(2, all_surfs, tag=1)
            gmsh.model.setPhysicalName(2, 1, "solid")

        # ── Periodic constraints ──────────────────────────────────────────
        # Affine transform bottom→top: translate (0, 1, 0)
        bt_transform = [1, 0, 0, 0,  0, 1, 0, 1,  0, 0, 1, 0,  0, 0, 0, 1]
        # Affine transform left→right: translate (1, 0, 0)
        lr_transform = [1, 0, 0, 1,  0, 1, 0, 0,  0, 0, 1, 0,  0, 0, 0, 1]

        # Sort by shared coordinate so master/slave lists align
        for e in range(4):
            boundary_curves[e].sort(key=lambda x: x[1])

        n_bot = len(boundary_curves[EDGE_BOTTOM])
        n_top = len(boundary_curves[EDGE_TOP])
        if n_bot == n_top and n_bot > 0:
            try:
                gmsh.model.mesh.setPeriodic(
                    1,
                    [c[0] for c in boundary_curves[EDGE_TOP]],     # slave
                    [c[0] for c in boundary_curves[EDGE_BOTTOM]],  # master
                    bt_transform,
                )
            except Exception:
                pass  # proceed without periodicity

        n_lft = len(boundary_curves[EDGE_LEFT])
        n_rgt = len(boundary_curves[EDGE_RIGHT])
        if n_lft == n_rgt and n_lft > 0:
            try:
                gmsh.model.mesh.setPeriodic(
                    1,
                    [c[0] for c in boundary_curves[EDGE_RIGHT]],  # slave
                    [c[0] for c in boundary_curves[EDGE_LEFT]],   # master
                    lr_transform,
                )
            except Exception:
                pass

        # ── Generate mesh ─────────────────────────────────────────────────
        gmsh.model.mesh.generate(2)
        gmsh.write(msh_path)

    except Exception:
        gmsh.finalize()
        return None

    gmsh.finalize()
    return msh_path


def convert_msh_to_xdmf(msh_path: str) -> str | None:
    """Convert a Gmsh .msh file to .xdmf/.h5 (DOLFINx-compatible).

    Extracts only triangle cells and drops the z-coordinate.
    """
    try:
        mesh = meshio.read(msh_path)
    except Exception:
        return None

    tri_cells = [c for c in mesh.cells if c.type == "triangle"]
    if not tri_cells:
        return None

    points_2d = mesh.points[:, :2]
    xdmf_path = msh_path.replace(".msh", ".xdmf")
    tri_mesh = meshio.Mesh(points=points_2d, cells=tri_cells)
    meshio.xdmf.write(xdmf_path, tri_mesh)
    return xdmf_path


print("Gmsh meshing functions defined.")

Gmsh meshing functions defined.


In [63]:
# ──────────────────────────────────────────────────────────────────────────────
# 11. Batch Mesh Generation, Save Dataset & Convert to XDMF
# ──────────────────────────────────────────────────────────────────────────────

# Load the previously saved dataset
if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(
        f"Could not find {DATASET_PATH}. Run the dataset save cell first."
    )
data = np.load(DATASET_PATH, allow_pickle=True)
print(f"Loaded dataset: {len(data['grf'])} samples")

n_samples = len(data["grf"])
mesh_ok = 0
mesh_fail = 0
xdmf_ok = 0

for i in range(n_samples):
    grf_i = data["grf"][i]
    th_i = float(data["threshold"][i])

    # Generate mesh
    msh_path = mesh_single_sample(grf_i, th_i, i)
    if msh_path is None:
        mesh_fail += 1
        continue
    mesh_ok += 1

    # Convert .msh → .xdmf
    xdmf_path = convert_msh_to_xdmf(msh_path)
    if xdmf_path is not None:
        xdmf_ok += 1

    if (i + 1) % 20 == 0:
        print(f"  [{i+1:3d}/{n_samples}]  mesh OK={mesh_ok}  fail={mesh_fail}  xdmf={xdmf_ok}")

print(f"\n{'='*60}")
print(f"Meshing complete: {mesh_ok}/{n_samples} meshes, {xdmf_ok} XDMF files")
print(f"Failed: {mesh_fail}")

Loaded dataset: 10 samples











Meshing complete: 10/10 meshes, 10 XDMF files
Failed: 0


In [64]:
# ──────────────────────────────────────────────────────────────────────────────
# 12. Mesh Visualization & Verification
# ──────────────────────────────────────────────────────────────────────────────

def plot_mesh_gallery(
    data: dict,
    mesh_dir: str = MESH_DIR,
    n_show: int = 6,
    figsize: tuple = (15, 10),
) -> None:
    """Plot sample meshes overlaid on their binary maps."""
    indices = np.linspace(0, len(data["grf"]) - 1, n_show, dtype=int)
    ncols = 3
    nrows = (n_show + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = axes.flatten()

    for ax_idx, sample_idx in enumerate(indices):
        ax = axes[ax_idx]
        binary = data["binary"][sample_idx]
        ax.imshow(
            binary.T, origin="lower", extent=[0, 1, 0, 1],
            cmap="gray", alpha=0.3,
        )

        msh_file = os.path.join(mesh_dir, f"sample_{sample_idx:04d}.msh")
        if os.path.exists(msh_file):
            mesh = meshio.read(msh_file)
            pts = mesh.points[:, :2]
            for cell_block in mesh.cells:
                if cell_block.type == "triangle":
                    tri = cell_block.data
                    ax.triplot(pts[:, 0], pts[:, 1], tri,
                               linewidth=0.3, color="steelblue")
            ax.set_title(f"Sample {sample_idx} ({len(pts)} nodes)", fontsize=10)
        else:
            ax.set_title(f"Sample {sample_idx} (no mesh)", fontsize=10)

        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_aspect("equal")

    for ax in axes[n_show:]:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig("mesh_gallery.png", dpi=150)
    plt.show()


# ── Plot gallery ──────────────────────────────────────────────────────────────
plot_mesh_gallery(data)

# ── Mesh statistics ───────────────────────────────────────────────────────────
print("\nMesh statistics:")
node_counts = []
elem_counts = []
for i in range(len(data["grf"])):
    msh_file = os.path.join(MESH_DIR, f"sample_{i:04d}.msh")
    if not os.path.exists(msh_file):
        continue
    m = meshio.read(msh_file)
    node_counts.append(len(m.points))
    for cb in m.cells:
        if cb.type == "triangle":
            elem_counts.append(len(cb.data))
            break

if node_counts:
    print(f"  Nodes:    min={min(node_counts)}, max={max(node_counts)}, "
          f"mean={np.mean(node_counts):.0f}")
    print(f"  Elements: min={min(elem_counts)}, max={max(elem_counts)}, "
          f"mean={np.mean(elem_counts):.0f}")

# ── Verify periodic node matching ─────────────────────────────────────────────
print("\nPeriodic node verification (first 5 meshes):")
for i in range(min(5, len(data["grf"]))):
    msh_file = os.path.join(MESH_DIR, f"sample_{i:04d}.msh")
    if not os.path.exists(msh_file):
        continue
    m = meshio.read(msh_file)
    pts = m.points[:, :2]
    tol = 1e-4

    # Find boundary nodes
    bot = pts[np.abs(pts[:, 1]) < tol]
    top = pts[np.abs(pts[:, 1] - 1) < tol]
    lft = pts[np.abs(pts[:, 0]) < tol]
    rgt = pts[np.abs(pts[:, 0] - 1) < tol]

    # Check matching: sort by shared coordinate
    bot_x = np.sort(bot[:, 0])
    top_x = np.sort(top[:, 0])
    lft_y = np.sort(lft[:, 1])
    rgt_y = np.sort(rgt[:, 1])

    bt_match = (len(bot_x) == len(top_x) and
                np.allclose(bot_x, top_x, atol=1e-3) if len(bot_x) > 0 else True)
    lr_match = (len(lft_y) == len(rgt_y) and
                np.allclose(lft_y, rgt_y, atol=1e-3) if len(lft_y) > 0 else True)

    status = "✓" if (bt_match and lr_match) else "✗"
    print(f"  Sample {i}: B={len(bot_x)} T={len(top_x)} L={len(lft_y)} R={len(rgt_y)} "
          f"  BT={'match' if bt_match else 'MISMATCH'} "
          f"LR={'match' if lr_match else 'MISMATCH'} {status}")

# ── Verify XDMF round-trip ───────────────────────────────────────────────────
print("\nXDMF round-trip check:")
for i in range(min(3, len(data["grf"]))):
    xdmf_file = os.path.join(MESH_DIR, f"sample_{i:04d}.xdmf")
    if os.path.exists(xdmf_file):
        xm = meshio.read(xdmf_file)
        print(f"  Sample {i}: {len(xm.points)} nodes, "
              f"{sum(len(cb.data) for cb in xm.cells)} elements ✓")
    else:
        print(f"  Sample {i}: no XDMF file")

/tmp/ipykernel_122922/385375696.py:47: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



Mesh statistics:










  Nodes:    min=638, max=1193, mean=905
  Elements: min=10, max=2226, mean=1346

Periodic node verification (first 5 meshes):

  Sample 0: B=16 T=16 L=8 R=8   BT=match LR=match ✓

  Sample 1: B=15 T=15 L=15 R=15   BT=match LR=match ✓

  Sample 2: B=20 T=20 L=17 R=18   BT=MISMATCH LR=MISMATCH ✗

  Sample 3: B=25 T=25 L=13 R=13   BT=match LR=match ✓

  Sample 4: B=0 T=0 L=16 R=16   BT=match LR=match ✓

XDMF round-trip check:
  Sample 0: 973 nodes, 1768 elements ✓
  Sample 1: 1193 nodes, 2226 elements ✓
  Sample 2: 999 nodes, 1787 elements ✓
